# 01 · Glitching — voltage glitch (Crowbar)

**Voltage glitching** briefly shorts the target's power rail with a MOSFET
(the *crowbar*), dropping VCC for a few nanoseconds at a precise moment to
make the CPU skip or corrupt an instruction. On FaultyCat this is the
`cat.crowbar` engine.

> ⚠️ Crowbar shorts the target's VCC. Wire the crowbar output to the
> target's power line and share GND. Keep the shield on.

In [ ]:
import faultycat as fc

# --- edit for your target ---
SIM      = False           # True to dry-run without a board
OUTPUT   = 'lp'            # crowbar output path: 'lp' (low power) or 'hp'
RST_GP   = None           # GP wired to target nRST; set once firmware has 'reset'

cat = fc.connect(simulator=SIM)
cat.crowbar

## 1 · A single glitch (external trigger)

Configure by attribute, then `arm()` → `fire()`. With an external trigger
the engine enters `WAITING` and discharges when the target's trigger edge
arrives — with nothing wired it waits safely (no discharge). `last_fire_at_ms
> 0` in the status means it actually fired.

In [ ]:
cat.crowbar.trigger  = 'ext_rising'
cat.crowbar.output   = OUTPUT
cat.crowbar.delay_us = 50
cat.crowbar.width_ns = 100
cat.crowbar.arm()
cat.crowbar.status          # expect state=ARMED

In [ ]:
try:
    cat.crowbar.fire(trigger_timeout_ms=5000)   # waits for the external edge
except fc.EngineError as e:
    print('engine:', e)      # e.g. TRIGGER_TIMEOUT if no edge arrived
print(cat.crowbar.status)
cat.crowbar.disarm()

## 2 · Hunting the glitch — **you** program success

Whether a glitch *worked* is target-specific — the tool never guesses it.
This is exactly ChipWhisperer's model: loop over parameters, glitch, **read
the target, and classify each attempt** into groups you choose (success /
reset / normal) with a condition you write. (CW does this with
`GlitchController` + `gc.add("success")`.)

Edit `classify()` for your target. Here the loop uses `immediate` trigger
and reads the target UART after each shot; swap in your own tell (a return
value, an SWD read, a status byte).

In [ ]:
def classify(resp: bytes) -> str:
    """YOUR success condition — the whole point lives here."""
    if b'root' in resp or b'OK' in resp:   # the tell that a check was skipped
        return 'success'
    if not resp:
        return 'reset'
    return 'normal'

In [ ]:
gc = fc.GlitchController(['delay', 'width'])
gc.set_range('delay', range(0, 200, 20)).set_range('width', range(50, 400, 50))

if cat.uart:
    cat.uart.open()

for p in gc.glitch_values():
    cat.crowbar.trigger  = 'immediate'
    cat.crowbar.output   = OUTPUT
    cat.crowbar.delay_us = p['delay']
    cat.crowbar.width_ns = p['width']
    if RST_GP is not None:
        cat.target_reset(RST_GP)
    if cat.uart:
        cat.uart.reset_input()
    cat.crowbar.arm(); cat.crowbar.fire()
    resp = cat.uart.read_until(b'\n', timeout=0.1) if cat.uart else b''
    gc.add(classify(resp))

gc.counts()

In [ ]:
gc.plot(x='delay', y='width');

> **Faster, coarser alternative:** `cat.campaign('crowbar')` sweeps the grid
> *on-device* and streams `fire_status` / `verify_status` per step. Those are
> firmware-side fields — useful only if you wired the on-device verify hook;
> they are **not** your success. For real success, observe the target as
> above.

In [ ]:
cat.close()